# Phase 3: Preprocessing Pipeline

## Objective
Transform raw extracted text into structured, LLM-ready chunks.

## Pipeline Steps
1. Text Cleaning
2. Sentence-aware Chunking
3. Metadata Enrichment
4. Structured Output Generation

## Input
- JSON file from `data/interim/`
  - text
  - metadata

## Output
- JSON file in `data/processed/`
- Chunk artifacts in `artifacts/chunks/`

## Key Design Principles
- Config-driven chunking (chunk_size, overlap)
- Sentence-safe splitting
- Lightweight processing (no heavy NLP libs)
- Modular and production-ready

In [1]:
import os
import sys

# Add project root to path
sys.path.append(os.path.abspath(".."))

In [2]:
from src.preprocessing.pipeline import PreprocessingPipeline
from src.utils.config import BASE_DIR
from src.utils.helpers import load_json
from src.utils.logger import get_logger


logger = get_logger("PreprocessingNotebook")
logger.info("Notebook started")

2026-04-22 22:38:29,182 - PreprocessingNotebook - INFO - Notebook started


## Load Input File

We use the output from Phase 2 (Data Ingestion), stored in:
`data/interim/`

In [3]:
input_file = os.path.join(
    BASE_DIR,
    "data",
    "interim",
    "ERIA_Sample_1.json"
)

print("Input File:", input_file)

Input File: F:\DATA SCIENCE\Projects\Education Regulation Impact Analyzer (ERIA)\data\interim\ERIA_Sample_1.json


## Load JSON Data

In [4]:
data = load_json(input_file)

print("Loaded keys:", data.keys())
print("Text length:", len(data["text"]))

Loaded keys: dict_keys(['text', 'metadata'])
Text length: 39227


## Initialize Preprocessing Pipeline

This pipeline internally performs:
- Cleaning
- Chunking
- Metadata enrichment
- Saving outputs

In [5]:
pipeline = PreprocessingPipeline()

file_name = os.path.basename(input_file)

output = pipeline.process(input_file)

2026-04-22 22:38:29,311 - src.preprocessing.pipeline - INFO - Processing file: F:\DATA SCIENCE\Projects\Education Regulation Impact Analyzer (ERIA)\data\interim\ERIA_Sample_1.json
2026-04-22 22:38:29,313 - src.preprocessing.pipeline - INFO - Original text length: 39227
2026-04-22 22:38:29,346 - src.preprocessing.pipeline - INFO - Cleaned text length: 35092
2026-04-22 22:38:29,348 - src.preprocessing.chunker - INFO - Chunk 0 created
2026-04-22 22:38:29,349 - src.preprocessing.chunker - INFO - Chunk 1 created
2026-04-22 22:38:29,350 - src.preprocessing.chunker - INFO - Chunk 2 created
2026-04-22 22:38:29,351 - src.preprocessing.chunker - INFO - Chunk 3 created
2026-04-22 22:38:29,352 - src.preprocessing.chunker - INFO - Chunk 4 created
2026-04-22 22:38:29,353 - src.preprocessing.chunker - INFO - Chunk 5 created
2026-04-22 22:38:29,354 - src.preprocessing.chunker - INFO - Chunk 6 created
2026-04-22 22:38:29,354 - src.preprocessing.chunker - INFO - Chunk 7 created
2026-04-22 22:38:29,355 -

## Metadata Output

Includes:
- file_name
- num_pages
- extracted_at
- processed_at
- text_length
- num_chunks_estimated

In [6]:
print("Metadata:\n")
for key, value in output["metadata"].items():
    print(f"{key}: {value}")

Metadata:

file_name: ERIA_Sample_1.pdf
num_pages: 19
extracted_at: 2026-04-19T19:57:38.335506
text_length: 35092
processed_at: 2026-04-22T17:08:29.406021
num_chunks: 55


## Chunk Summary

- Total chunks created
- Ensures LLM-friendly size
- Overlap maintains context continuity

In [7]:
print("Text length:", len(output["chunks"][0]["text"]))
print("Total chunks:", len(output["chunks"]))

Text length: 734
Total chunks: 55


In [8]:
num_chunks = len(output["chunks"])
print("Total Chunks:", num_chunks)

Total Chunks: 55


In [9]:
chunk_sizes = [len(c["text"]) for c in output["chunks"]]

print("Max:", max(chunk_sizes))
print("Min:", min(chunk_sizes))
print("Avg:", sum(chunk_sizes) // len(chunk_sizes))

Max: 1061
Min: 537
Avg: 728


## Sample Chunk

Each chunk contains:
- chunk_id
- text

In [10]:
sample_chunk = output["chunks"][0]

print("Chunk ID:", sample_chunk["chunk_id"])
print("\nText Preview:\n")
print(sample_chunk["text"][:500])

Chunk ID: 0

Text Preview:

ogiat -Pears fagaaa frataenaa ayer ART University Grants Commission area : (TSreN FATA, AKA AHI) Prof. .6-1/2026(interniship) 26 7, 1948 / 16 April, 2026 Subject: Information regarding Guidelines the Prime Minister Internship Scheme (PMIS) aretha agiqa / Aetea, The Ministry Corporate Affairs has launched the Prime Ministers Internship Scheme (PMIS) provide structured internship opportunities the youth top companies and provide exposure real-life business environment across varied professions and


## Overlap Validation

Check if consecutive chunks share context (overlap region).

In [11]:
for i in range(10):
    if len(output["chunks"]) > 1:
        chunk1 = output["chunks"][i]["text"]
        chunk2 = output["chunks"][i + 1]["text"]

        print("End of Chunk 1:\n", chunk1[-100:])
        print("\nStart of Chunk 2:\n", chunk2[:100])

End of Chunk 1:
 rticipate PMIS. The eligible candidates can apply the official portal: https://pminternship.mca.gov.

Start of Chunk 2:
 /login/ The Guidelines the Prime Minister Internship Scheme (PMIS) are attached. therefore requested
End of Chunk 1:
 s announced the Budget 2024-25 with the aim providing internship opportunities youth, Top Companies.

Start of Chunk 2:
 s announced the Budget 2024-25 with the aim providing internship opportunities youth, Top Companies.
End of Chunk 1:
  has now been extended till December 2026, with target providing 1.10 lakh internship opportunities.

Start of Chunk 2:
 has now been extended till December 2026, with target providing 1.10 lakh internship opportunities. 
End of Chunk 1:
 ct, are not treated employees clarified the Ministry Labor and Employment vide its dated 27.09.2024.

Start of Chunk 2:
 ct, are not treated employees clarified the Ministry Labor and Employment vide its dated 27.09.2024.
End of Chunk 1:
 clarified that internship d

In [12]:
raw_len = len(data["text"])
clean_len = output["metadata"]["text_length"]

print("Raw:", raw_len)
print("Clean:", clean_len)
print("Reduction %:", round((raw_len - clean_len) / raw_len * 100, 2))

Raw: 39227
Clean: 35092
Reduction %: 10.54


In [13]:
print("⚠️ TRUNCATION CHECK")

if raw_len <= 5000:
    print("WARNING: truncated (fix config.yaml)")
else:
    print("OK: full text")

⚠️ TRUNCATION CHECK
OK: full text


In [14]:
print(output["chunks"][-1]["text"][:500])

ted the participating Company/ Organization shall not permitted participate the pilot project again. 4.9 Enhancing Employability: Candidates are encouraged use the internship experience develop technical, professional, and soft skills relevant their career aspirations. Recognition performance: Interns will undergo continuous evaluation their performance and conduct the companies accordance with the companies policies. build confidence and create aspirational value for the Scheme, companies are e


## Output Storage Locations

### Processed Data
`data/processed/ERIA_Sample_1.json`

### Chunk Artifacts
`artifacts/chunks/ERIA_Sample_1.json`

In [8]:
processed_path = os.path.join(BASE_DIR, "data", "processed", "ERIA_Sample_1.json")
chunks_path = os.path.join(BASE_DIR, "artifacts", "chunks", "ERIA_Sample_1.json")

print("Processed Exists:", os.path.exists(processed_path))
print("Chunks Exists:", os.path.exists(chunks_path))

Processed Exists: True
Chunks Exists: True


## Key Observations

- Text successfully cleaned and normalized
- Chunking respects sentence boundaries
- Overlap preserves semantic continuity
- Output is structured and LLM-ready

## Ready for Next Phase

➡️ Classification using Hugging Face BART MNLI